KPI CALCULATION

In [2]:
import pandas as pd
from google.colab import files
uploaded = files.upload()
# Get the exact uploaded filename
filename = list(uploaded.keys())[0]
print("Uploaded file:", filename)
df = pd.read_csv(filename)
print("Dataset loaded successfully!")
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])

Saving hospital_cleaned (1).csv to hospital_cleaned (1).csv
Uploaded file: hospital_cleaned (1).csv
Dataset loaded successfully!
Rows: 45000
Columns: 39


KPI 1 — Total Admissions

In [3]:
total_admissions = df["admission_id"].nunique()
print("KPI 1 - Total Admissions:", total_admissions)

KPI 1 - Total Admissions: 45000


KPI 2 — Occupancy Rate

In [4]:
#Convert admission and discharge dates
df["admission_date"] = pd.to_datetime(
    df["admission_date"],
    errors="coerce"
)
df["discharge_date"] = pd.to_datetime(
    df["discharge_date"],
    errors="coerce"
)
# Calculate Length of Stay
df["Length_of_Stay"] = (
    df["discharge_date"] - df["admission_date"]
).dt.days
# Negative value checking
df.loc[df["Length_of_Stay"] < 0, "Length_of_Stay"] = 0

In [5]:
# Calculate total available beds
ward_beds = (
    df[["ward_id", "total_beds"]]
    .drop_duplicates()
)

total_beds = ward_beds["total_beds"].sum()

print("Total Available Beds:", total_beds)

Total Available Beds: 415


In [6]:
# Find the analysis period
start_date = df["admission_date"].min()
end_date = df["discharge_date"].max()

number_of_days = (
    end_date - start_date
).days + 1

print("Start Date:", start_date)
print("End Date:", end_date)
print("Observation Days:", number_of_days)

Start Date: 2020-01-01 00:00:00
End Date: 2026-01-12 00:00:00
Observation Days: 2204


In [7]:
# Calculate occupied bed-days
occupied_bed_days = df["Length_of_Stay"].sum()

In [8]:
# Final Occupancy Rate
occupancy_rate = (
    occupied_bed_days /
    (total_beds * number_of_days)
) * 100

print(
    "KPI 2 - Occupancy Rate:",
    round(occupancy_rate, 2),
    "%"
)

KPI 2 - Occupancy Rate: 25.36 %


KPI 3 — Average Length of Stay

In [9]:
average_length_of_stay = df["Length_of_Stay"].mean()
print(
    "KPI 3 - Average Length of Stay:",
    round(average_length_of_stay, 2),
    "days"
)

KPI 3 - Average Length of Stay: 5.16 days


KPI 4 — Readmission Rate

In [10]:
# Count admissions for each patient
patient_admission_counts = (
    df.groupby("patient_id")["admission_id"]
    .nunique())
# Patients with more than one admission
readmitted_patients = (
    patient_admission_counts[
        patient_admission_counts > 1])
# Total unique patients
total_unique_patients = df["patient_id"].nunique()
# Number of readmitted patients
total_readmitted_patients = len(readmitted_patients)
print("Total Unique Patients:", total_unique_patients)
print("Readmitted Patients:", total_readmitted_patients)

Total Unique Patients: 23275
Readmitted Patients: 13248


In [11]:
# Final readmission rate
readmission_rate = (
    total_readmitted_patients /
    total_unique_patients
) * 100

print(
    "KPI 4 - Readmission Rate:",
    round(readmission_rate, 2),
    "%"
)

KPI 4 - Readmission Rate: 56.92 %


KPI 5 — Bed Utilization Rate

In [12]:
bed_utilization_rate = (
    occupied_bed_days /
    (total_beds * number_of_days)
) * 100
print(
    "KPI 5 - Bed Utilization Rate:",
    round(bed_utilization_rate, 2),
    "%")

KPI 5 - Bed Utilization Rate: 25.36 %


KPI 6 — Department Efficiency Score

In [13]:
# Admissions by department
department_admissions = (
    df.groupby("department_id")["admission_id"].nunique().reset_index(name="Admissions"))
# Staff by department
department_staff = (df.groupby("department_id")["Employee_Count"].max().reset_index(name="Staff_Count"))

In [14]:
# Combining
department_efficiency = department_admissions.merge(department_staff,on="department_id",how="left")
display(department_efficiency)

,department_id,Admissions,Staff_Count
0,1,8777,51
1,2,7695,54
2,3,10126,43
3,4,8438,33
4,5,5924,40
5,6,4040,45


In [16]:
# Admissions per staff
department_efficiency["Admissions_per_Staff"] = (
    department_efficiency["Admissions"] /
    department_efficiency["Staff_Count"])
max_value = department_efficiency["Admissions_per_Staff"].max()

In [17]:
# Departmentwise score
department_efficiency["Efficiency_Score"] = (
    department_efficiency["Admissions_per_Staff"] /max_value) * 100
department_efficiency["Efficiency_Score"] = (
    department_efficiency["Efficiency_Score"].round(2))
display(
    department_efficiency[["department_id", "Efficiency_Score"]])

,department_id,Efficiency_Score
0,1,67.31
1,2,55.73
2,3,92.10
3,4,100.00
4,5,57.92
5,6,35.11


In [18]:
#  Total admissions in the hospital
total_admissions = df["admission_id"].nunique()
# Total staff in the hospital
total_staff = df["Employee_Count"].max()
# Overall admissions per staff
overall_admissions_per_staff = (
    total_admissions / total_staff)
print("Overall Admissions per Staff:",
      round(overall_admissions_per_staff))

Overall Admissions per Staff: 833


In [19]:
# Highest department efficiency
highest_department_efficiency = (
    department_efficiency["Admissions_per_Staff"].max())
# Overall hospital efficiency score
overall_efficiency_score = (
    department_efficiency["Efficiency_Score"].mean()).round(2)
print(
    "Overall Hospital Department Efficiency Score:",
    overall_efficiency_score,"%")

Overall Hospital Department Efficiency Score: 68.03 %


In [24]:
# DEPARTMENT-WISE KPI TABLE
department_kpis = (
    df.groupby(
        ["department_id", "department_name"]
    )
    .agg(
        Admissions=("admission_id", "nunique"),
        Staff_Count=("Employee_Count", "max")
    )
    .reset_index()
)

# Admissions per Staff
department_kpis["Admissions_per_Staff"] = (
    department_kpis["Admissions"] /
    department_kpis["Staff_Count"]
).round(2)

# Relative Efficiency Score (0–100)
max_efficiency = department_kpis[
    "Admissions_per_Staff"
].max()

department_kpis["Efficiency_Score"] = (
    department_kpis["Admissions_per_Staff"] /
    max_efficiency * 100
).round(2)

display(department_kpis)

,department_id,department_name,Admissions,Staff_Count,Admissions_per_Staff,Efficiency_Score
0,1,Emergency,8777,51,172.10,67.31
1,2,Internal Medicine,7695,54,142.50,55.73
2,3,Surgery,10126,43,235.49,92.10
3,4,Pediatrics,8438,33,255.70,100.00
4,5,Orthopedics,5924,40,148.10,57.92
5,6,Icu,4040,45,89.78,35.11


In [25]:
with pd.ExcelWriter(
    "/content/hospital_final_dataset_with_kpis.xlsx",
    engine="openpyxl"
) as writer:
    df.to_excel(
        writer,
        sheet_name="Final_Dataset",
        index=False
    )
    kpi_summary.to_excel(
        writer,
        sheet_name="KPI_Summary",
        index=False
    )
    department_kpis.to_excel(
        writer,
        sheet_name="Department_KPIs",
        index=False
    )
files.download(
    "/content/hospital_final_dataset_with_kpis.xlsx"
)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>